In [ ]:
from dataclasses import dataclass
from openai import OpenAI
import os
from openai.types.chat import ChatCompletionMessageParam


@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    
    Provider(
        "Groq",
        "GROQ_API_KEY",
        True,
        "https://api.groq.com/openai/v1",
        "openai/gpt-oss-120b"
    ),
    Provider(
        "OpenRouter",
        "OPENROUTER_API_KEY",
        True,
        "https://openrouter.ai/api/v1",
        "openai/gpt-sol-latest"
    ),
    Provider(
        "OpenAI",
        "OPENAI_API_KEY",
        True,
        None,
        "gpt-4o-mini"
    ),
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(
        f"No provider key set, add one of {expected} "
        "to your environment variables"
    )

def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)

    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")


def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": prompt,
        }]
    )

    return result.choices[0].message.content or ""

ZERO SHOT COT -----------

In [9]:
ZERO_SHOT_COT_SUFFIX = "\n\nLet's think step by step."

def zero_shot_cot_prompt(question: str) -> str:
    """This function takes a question and returns a prompt for the zero-shot COT model."""
    return f"Question: {question}{ZERO_SHOT_COT_SUFFIX}"

In [10]:
QUESTION = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [ ]:
print(zero_shot_cot_prompt(QUESTION))

In [ ]:
reply = llm_reply(zero_shot_cot_prompt(QUESTION))

print(reply)

FEW SHOT COT --------

In [15]:
FEW_SHOT_EXAMPLES = """
Q: There are 15 trees in the grove. Grove workers will plant 6 trees today. How many trees will be in the grove?
A: There are 15 trees to start. After planting 6 more, there are 15 + 6 = 21 trees.
Final answer: 21

Q: Leah had 32 chocolates and her sister had 42. They ate 35. How many pieces do they have left in total?
A: Leah and her sister had 32 + 42 = 74 chocolates together. After eating 35, they have 74 - 35 = 39 left.
Final answer: 39
""".strip()

In [16]:
def few_shot_cot_prompt(question: str) -> str:
    """This function takes a question and returns a prompt for the few-shot COT model."""
    return f"""Solve each question by reasoning step by step.
    {FEW_SHOT_EXAMPLES}
    
    Question: {question}
    Answer:
    """

In [ ]:
reply = llm_reply(few_shot_cot_prompt(QUESTION))

print(reply)

In [18]:
TRICKY = (
    "A juggler can juggle 16 balls. Half of the balls are golf balls, and "
    "half of the golf balls are blue. How many blue golf balls are there?"
)

In [ ]:
reply = llm_reply(zero_shot_cot_prompt(TRICKY))

print(reply)

In [ ]:
reply = llm_reply(few_shot_cot_prompt(TRICKY))

print(reply)